# ⚡ Reaching 88 MPH
## An assistant that reviews your valuations

**Section 9** of *Search, Extract & Value: The Lightning Fast Assistant* 🚗💨

---


> ### ☁️ This is the **hosted** version.
> The other version is **`09_88mph_local.ipynb`**.
>
> 📌 **The two notebooks reach the same answer, and that is worth noticing.**
> Sections 5 and 7 both ended with the big hosted model and the small local
> one behaving differently. This one does not — because the statistics are
> not the model's job. See the last act.

### What you need

| | |
|---|---|
| 🔑 **An API key** | `ANTHROPIC_API_KEY` in your `.env`. |
| 🌐 **Internet** | yes — every question is a call to Anthropic. |
| 💰 **Cost** | a few cents. Defaults to Haiku, the cheapest model. |
| 🎮 **A GPU** | **not needed.** That is the whole point of this variant. |


---

Every appraiser reading this has signed off on a countywide ratio study. One
median, one COD, one PRD, all three inside the IAAO standard, and the year
closed out.

This notebook is about what that number hides.

Hill Valley County passes every IAAO standard. **Three of its six
neighborhoods do not** — and no two of them fail for the same reason. One is
assessed too low. One is wildly inconsistent. One is quietly unfair to cheaper
homes. Three different problems, three different afternoons of work, and the
countywide figure shows none of them.

The finding is not the clever part. **The finding is arithmetic, and we do it
with no AI at all in Acts 1 and 2.** What the assistant adds is that you can
*ask* for it — in a sentence, in any slice you like, without waiting on
anybody to write you a report.

There is one idea underneath all of it, and it is the thing to take home:

> ### The agent decides **what to ask**. The tool decides **what is correct**.

Keep an eye on that as it goes. Act 4 is where it earns its keep.

---

# 🚦 LIVE DEMO PATH

## Setup

**This is the only cell where the two versions of this notebook differ.**
Everything below it is identical, because both versions import the same tools
and the same graph from `src/delorean/`.

In [1]:
import json

from delorean import agent, data, llm, prompts, ratio, tools, viz
from delorean.tools import query, review

# Runs on Anthropic's machines. Needs ANTHROPIC_API_KEY in your .env.
client = llm.get_api_client()
respond = agent.api_responder(client, llm.api_model())

data.check()

---

## Act 1 — 🧮 The arithmetic, with no AI in the room

Before any model gets involved, here is the whole of a ratio study.

An assessment **ratio** is one division: what we said a property was worth,
over what it actually sold for.

$$\text{ratio} = \frac{\text{assessed value}}{\text{sale price}}$$

One sale tells you nothing — anybody can miss one house. A hundred sales tell
you three separate things, and IAAO gives each of them its own test.

In [2]:
# Five real sales out of Hilldale, and the ratio for each one.
feed = data.sales_feed_df()
hilldale = feed[(feed.neighborhood == "Hilldale") & (feed.arms_length == 1)]

sample = hilldale.head(5)[["parcel_id", "sale_price", "assessed_value"]].copy()
sample["ratio"] = sample.assessed_value / sample.sale_price

print("What we said, over what it sold for:\n")
for row in sample.itertuples():
    print(f"  {row.parcel_id}   sold ${row.sale_price:>9,}   "
          f"assessed ${row.assessed_value:>9,}   ratio {row.ratio:.3f}")

What we said, over what it sold for:

  04-490-2882   sold $  378,000   assessed $  351,400   ratio 0.930
  09-357-5765   sold $  272,500   assessed $  311,500   ratio 1.143
  04-238-5689   sold $  497,000   assessed $  397,200   ratio 0.799
  07-139-1887   sold $  367,000   assessed $  363,200   ratio 0.990
  07-445-3526   sold $  381,000   assessed $  322,400   ratio 0.846


Now the three tests. Each one asks a different question, and a neighborhood
can pass any of them while failing the others.

| | Measures | The question it answers | IAAO wants |
|---|---|---|---|
| **Median ratio** | Level | *Are we high or low overall?* | 0.90 – 1.10 |
| **COD** | Uniformity | *Are we consistent from house to house?* | 5 – 15 |
| **PRD** | Equity | *Do cheap and expensive homes get the same deal?* | 0.98 – 1.03 |

⚠️ **The median, not the average.** A couple of odd sales drag an average
around; the median barely moves. Everything below that could be either is the
median on purpose — and in Act 4 you will watch a language model reach
straight for `AVG()`.

In [3]:
# `ratio.py` is pure arithmetic -- no model, no database, no magic.
# You can check every number in it by hand.
result = ratio.ratio_stats(hilldale)

for test in result["tests"]:
    print(f"  {test['test']:<11} {test['metric']:<13} {test['value']:>8}"
          f"   standard {test['standard']:<14} {test['verdict']}")

print()
print("  ", result["summary"])

  Level       Median ratio    0.9213   standard 0.9 to 1.1     PASS
  Uniformity  COD            13.6257   standard 5.0 to 15.0    PASS
  Equity      PRD             1.0431   standard 0.98 to 1.03   FAIL

   FAILS Equity (PRD) on 50 sales: Median ratio 0.9213 (PASS); COD 13.6257 (PASS); PRD 1.0431 (FAIL).


---

## Act 2 — 🚨 The county passes. Half of it does not.

Same arithmetic, run twice. First over the whole county, then one
neighborhood at a time.

**There is still no AI anywhere in this notebook.**

In [4]:
countywide = ratio.ratio_stats(feed)
viz.show_scorecard(countywide, title="Hill Valley County — every qualified sale")

Three green badges. A clean year. This is the number that goes in the annual
report, and it is entirely honest.

Now break it apart.

In [5]:
by_hood = ratio.ratio_by_group(feed, "neighborhood")
viz.show_ratio_table(by_hood.to_dict(orient="records"),
                     title="The same sales, one neighborhood at a time")

neighborhood,sales,Median ratio,COD,PRD,verdict
Courthouse Square,30,0.926,✗ 22.08,1.028,✗ FAIL
Hilldale,50,0.921,13.63,✗ 1.043,✗ FAIL
Riverside Heights,32,✗ 0.809,6.21,1.001,✗ FAIL
Eastwood Ravine,27,0.984,6.06,1.001,✓ PASS
Lyon Estates,42,0.948,6.04,1.001,✓ PASS
Twin Pines Ranch,29,0.929,9.33,1.002,✓ PASS


### 👀 Look at *which column* is red

Not three failing neighborhoods — **three different failures**:

- **Riverside Heights** fails on **Level**. The market ran away and the roll
  did not follow. Every owner there is under-assessed, which means everyone
  else in the county is quietly covering their share.
- **Courthouse Square** fails on **Uniformity**. The median is fine. The
  *spread* is enormous — historic housing stock, valued inconsistently. This
  is the neighborhood generating appeals.
- **Hilldale** fails on **Equity**. Cheaper homes carry a higher ratio than
  expensive ones. Regressive, and the hardest of the three to see without
  running the numbers.

One of those needs a revaluation. One needs a model rebuild. One needs a
closer look at how the low end is being valued. **The countywide figure would
have sent you home.**

---

## Act 3 — 🗣️ Now make it askable

Everything so far required somebody who knows pandas and knows where the data
lives. That person is busy.

So we hand a model the ability to query the roll — and the interesting part is
how little that takes. This is the entire briefing it gets:

In [6]:
print(query.SCHEMA_HELP)

Two tables, both in the Appraiser's office database. They do NOT need to be
joined -- sales_feed already carries the parcel characteristics you need.

TABLE parcels -- every parcel in the county (1,200 rows)
  parcel_id         TEXT     'NN-NNN-NNNN', e.g. '04-127-3891'
  situs_address     TEXT     street address of the property
  owner_name        TEXT     name on the roll, e.g. 'Biff Tannen'
  neighborhood      TEXT     one of: Courthouse Square, Eastwood Ravine, Hilldale, Lyon Estates, Riverside Heights, Twin Pines Ranch
  property_class    TEXT     'R01' single family, 'R02' condominium,
                             'C01' commercial, 'V01' vacant land
  year_built        INTEGER  year of construction
  living_area_sqft  INTEGER  heated living area
  lot_size_sqft     INTEGER  size of the lot
  assessed_value    INTEGER  the value we assessed, in dollars
  latitude          REAL     needed to put a parcel on a map
  longitude         REAL

TABLE sales_feed -- sales sent over from th

Two tables, **23 columns between them** — 11 and 12 — in plain English, curated down from the 29 the database really holds. Note what is **not** there:
`parcels` really has sixteen columns, and the model is shown eleven. Every
extra column is one more thing to reach for and get wrong.

Note also what it cannot reach. The Clerk's deeds and the Tax Collector's
balances are separate offices with separate systems — that gap is Section 10's
whole subject, and no amount of asking nicely will cross it from here.

Now give it one tool and ask it something.

In [7]:
sql_only = agent.build_agent(respond, ["run_sql"], max_steps=3)

state = agent.ask(sql_only,
                  "Which five parcels in the county have the highest "
                  "assessed value?",
                  system=prompts.REVIEW_ASSISTANT)

# The query is the model's own work. Nothing here wrote it for it.
for step in state["trace"]:
    if step["step"] == "act":
        print("The model wrote this SQL:\n")
        print("   ", step["args"]["sql"])
        print()

print(agent.final_answer(state))

The model wrote this SQL:

    SELECT parcel_id, owner_name, situs_address, assessed_value FROM parcels ORDER BY assessed_value DESC LIMIT 5

The five highest-assessed parcels in Hill Valley County are led by Ivan Baines' property at 1338 Elm St (04-377-4672) at $2,778,400, followed by Stella Evans at 3173 Maple Ave ($2,533,400), Keith Quinn at 2852 Lyon Dr ($2,171,900), Joey Hendricks at 501 Peabody Farm Rd ($2,068,100), and George Lindqvist at 1820 Hilldale Ct ($1,997,500).


That SQL is readable, checkable, and it ran against a **read-only** connection
that physically cannot write to your roll. When somebody in your office asks
*"you're letting an AI loose on the assessment roll?"* — this is the answer,
and it is four lines in [`tools/query.py`](../src/delorean/tools/query.py):
one statement, `SELECT` only, read-only connection, row cap.

---

## Act 4 — ⚠️ Why the statistics are a tool and not a prompt

Here is the obvious next thought: the model can write SQL, and a ratio study
is just arithmetic — so why not let it work the whole thing out itself?

Let's try exactly that. Same model, same one tool, asked as clearly as we can
manage.

In [8]:
naive = agent.ask(sql_only,
                  "Using SQL, work out the average assessment ratio "
                  "(assessed value over sale price) for each neighborhood, "
                  "arm's length sales only.",
                  system=prompts.REVIEW_ASSISTANT_NO_RATIO_TOOL)

for step in naive["trace"]:
    if step["step"] == "act":
        print("The model's SQL:\n")
        print("   ", step["args"]["sql"])
        print()
        rows = step["result"].get("data", {}).get("rows", [])
        for row in rows:
            print("   ", row)

The model's SQL:

    SELECT neighborhood, AVG(ratio) as average_ratio FROM sales_feed WHERE arms_length = 1 GROUP BY neighborhood ORDER BY neighborhood

    {'neighborhood': 'Courthouse Square', 'average_ratio': 0.9396333333333333}
    {'neighborhood': 'Eastwood Ravine', 'average_ratio': 0.9785962962962963}
    {'neighborhood': 'Hilldale', 'average_ratio': 0.937014}
    {'neighborhood': 'Lyon Estates', 'average_ratio': 0.9630357142857142}
    {'neighborhood': 'Riverside Heights', 'average_ratio': 0.7985875}
    {'neighborhood': 'Twin Pines Ranch', 'average_ratio': 0.9648724137931035}


### 🕵️ Nothing about that looks wrong

And that is the problem.

The SQL is clean. It filters to arm's length sales, groups by neighborhood,
and returns six numbers between 0.79 and 0.98 — exactly the range an appraiser
expects an assessment ratio to live in. Riverside Heights even comes out
lowest, which is correct. You could put this table in front of a board.

**It is still wrong**, in three ways that do not announce themselves:

1. **It is an average, not a median.** IAAO asks for the median precisely
   because a couple of odd sales drag a mean around. This is the single most
   common mistake in an amateur ratio study.
2. **There is no COD.** Nothing here says whether the assessments are
   *consistent* — the test Courthouse Square fails.
3. **There is no PRD.** Nothing here says whether cheap and expensive homes
   are treated alike — the test Hilldale fails.

Two of the three IAAO tests have quietly ceased to exist, and the third has
been replaced by a lookalike. Compare the numbers side by side:

In [9]:
truth = by_hood.set_index("neighborhood")["Median ratio"]

print(f"  {'Neighborhood':<20} {'model (average)':>15} {'IAAO (median)':>14}")
print("  " + "-" * 53)

for step in naive["trace"]:
    if step["step"] != "act":
        continue
    for row in step["result"].get("data", {}).get("rows", []):
        hood = row.get("neighborhood")
        if hood not in truth.index:
            continue
        # Whichever column it decided to call the ratio.
        guesses = [v for k, v in row.items()
                   if "ratio" in str(k).lower() and isinstance(v, (int, float))]
        said = f"{guesses[0]:.3f}" if guesses else "?"
        gap = f"{abs(guesses[0] - truth[hood]):.3f}" if guesses else ""
        print(f"  {hood:<20} {said:>15} {truth[hood]:>14.3f}"
              f"    off by {gap}")

  Neighborhood         model (average)  IAAO (median)
  -----------------------------------------------------
  Courthouse Square              0.940          0.926    off by 0.013
  Eastwood Ravine                0.979          0.984    off by 0.005
  Hilldale                       0.937          0.921    off by 0.016
  Lyon Estates                   0.963          0.948    off by 0.015
  Riverside Heights              0.799          0.809    off by 0.010
  Twin Pines Ranch               0.965          0.929    off by 0.036


### 🔁 Both notebooks print the same six numbers here

This is worth stopping on. We ran that identical cell against a **4B model on
a laptop** and against a **hosted model in the cloud**, and they wrote the same
query and returned the same six figures. Neither is being careless. `AVG` is a
perfectly reasonable reading of "average ratio".

So the failure is not a small-model failure, and it does not go away by
spending more money on a bigger one. **It is a domain-knowledge failure**, and
the domain knowledge — median not mean, plus COD, plus PRD, plus the
arm's-length rules — has to live somewhere the model cannot get it wrong.

> ⚠️ It gets worse if you ask *less* precisely. Asked simply to "run the ratio
> study", the local 4B reached for `AVG(assessed_value / sale_price)` — and
> SQLite integer division turned every ratio into a whole number first, so
> Riverside Heights came back as **`0.000`**: thirty-two properties assessed
> at nothing at all. That one you would catch. **The table above is the one
> you would not.**

> **So the arithmetic does not live in the prompt. It lives in a tool.**
> `ratio_study` computes the median rather than the average, applies the
> arm's-length rules, and checks the result against the IAAO bands — every
> time, the same way, whichever model is driving.

Same question, with that tool available:

In [10]:
answer = tools.dispatch("ratio_study", {
    "sql": "SELECT parcel_id FROM sales_feed",
    "group_by": "neighborhood",
})
payload = json.loads(answer)["data"]

print(payload["summary"])
print()
print("  excluded as not arm's length, automatically:",
      ratio.ratio_stats(feed)["sales_excluded_not_arms_length"], "sales")

3 of 6 neighborhoods fail at least one IAAO standard: Courthouse Square fails Uniformity (COD); Hilldale fails Equity (PRD); Riverside Heights fails Level (Median ratio).

  excluded as not arm's length, automatically: 34 sales


---

## Act 5 — 🔧 Three tools, one loop

Now put it together. The assistant gets three tools, and the two interesting
ones **take a SQL query as their argument**:

| Tool | What it does |
|---|---|
| `run_sql` | fetch rows from the roll |
| `ratio_study` | judge a set of sales against the IAAO standards |
| `map_parcels` | plot a set of parcels |

That one decision is what makes this open-ended instead of a menu. *"By
neighborhood"*, *"homes built after 1980"* and *"where we came in under 80% of
price"* are all the same tool call with a different `WHERE` clause. **Nobody
had to anticipate the question.**

And a system prompt that has stopped being decorative:

In [11]:
print(prompts.REVIEW_ASSISTANT)

You are the valuation review assistant for the Hill Valley County Assessor's Office.

Use run_sql to look up parcels, owners, addresses and sales.
Use ratio_study for ANY question about assessment level, uniformity or equity. Never work out a median, a COD or a PRD in SQL yourself - IAAO requires the median rather than the average, and the tool applies the arm's length rules for you.
Use map_parcels ONLY when someone asks to see properties on a map.
Answer definitions and general appraisal questions directly from your own knowledge. Do not use a tool for those.

When a tool returns a result, answer from it. Say plainly which IAAO standards passed and which failed.

Keep every answer to three sentences at most. The full figures are already on screen in front of the user, so give them the finding and what it means - do not read the whole table back to them.


Three measured jobs in that prompt, none of them cosmetic:

1. **It buys back restraint.** At three tools with no prompt, the model
   reaches for one when it should just answer. Naming when each tool applies
   fixed that — 6/6, with no loss of recall.
2. **It protects the statistics** — the *never do this in SQL* line is Act 4,
   written down.
3. **It halves the wall clock.** Telling it the figures are already on screen
   took a question from 39 seconds to 12. Same answer, same accuracy.

In [12]:
assistant = agent.build_agent(respond,
                              ["run_sql", "ratio_study", "map_parcels"],
                              max_steps=5)

print(assistant.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	reason(reason)
	act(act)
	__end__([<p>__end__</p>]):::last
	__start__ --> reason;
	act --> reason;
	reason -. &nbsp;stop&nbsp; .-> __end__;
	reason -.-> act;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



The same diagram, rendered — GitHub draws this straight from the notebook:

```mermaid
graph TD;
	__start__([__start__]):::first
	reason(reason)
	act(act)
	__end__([__end__]):::last
	__start__ --> reason;
	act --> reason;
	reason -. stop .-> __end__;
	reason -. wants a tool .-> act;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc
```

Two nodes and one decision, and it is the same picture Section 6 put on a
slide. `reason` asks the model what it wants; `act` runs it; the edge loops
back with the result; when nothing more is wanted, it stops. That is the whole
of it — about forty lines in
[`agent.py`](../src/delorean/agent.py), written out rather than imported,
so there is nothing here you cannot change.

### The three questions

Question one. The one every office asks in January.

In [13]:
q1 = agent.ask(assistant,
               "How did the county do on the ratio study this year?",
               system=prompts.REVIEW_ASSISTANT)

viz.show_trace(q1["trace"], answer=agent.final_answer(q1))

sales examined,244
sales used,210
sales excluded not arms length,34
sales excluded unusable,0
verdict,PASS
note,"34 sale(s) excluded as not arm's length, per IAAO - foreclosures and family transfers are not market evidence."
summary,Meets all three IAAO standards on 210 sales: Median ratio 0.9268 (PASS); COD 12.2184 (PASS); PRD 1.0262 (PASS).
stratification note,"This selection mixes property classes (C01, R01, R02). IAAO asks for separate studies per class. To limit it to houses, add: WHERE property_class IN ('R01','R02')"


Now the question the countywide figure was hiding.

In [14]:
q2 = agent.ask(assistant,
               "Run the assessment ratio study broken down by neighborhood.",
               system=prompts.REVIEW_ASSISTANT)

print(agent.final_answer(q2))

for step in q2["trace"]:
    if step["step"] == "act" and step["tool"] == "ratio_study":
        viz.show_ratio_table(step["result"]["data"]["results"],
                             title="What the assistant found")

Here's the assessment ratio study by neighborhood:

**Pass:** Eastwood Ravine, Lyon Estates, and Twin Pines Ranch all meet all three IAAO standards.

**Fail:** Three neighborhoods have issues—Courthouse Square has poor uniformity (COD 22.08, well above the 15 standard), Hilldale is inequitable between price points (PRD 1.04, above the 1.03 standard), and Riverside Heights is significantly underassessed (median ratio 0.81, below the 0.90 standard).


neighborhood,sales,Median ratio,COD,PRD,verdict
Courthouse Square,30,0.926,✗ 22.08,1.028,✗ FAIL
Hilldale,50,0.921,13.63,✗ 1.043,✗ FAIL
Riverside Heights,32,✗ 0.809,6.21,1.001,✗ FAIL
Eastwood Ravine,27,0.984,6.06,1.001,✓ PASS
Lyon Estates,42,0.948,6.04,1.001,✓ PASS
Twin Pines Ranch,29,0.929,9.33,1.002,✓ PASS


---

## Act 6 — 🗺️ Where is it?

*Riverside Heights fails on level* is a sentence. **Thirty dots in the wrong
colour is a place**, and it is what turns a statistic into a work order.

In [15]:
q3 = agent.ask(assistant,
               "Show me the sales in Riverside Heights on a map.",
               system=prompts.REVIEW_ASSISTANT)

print(agent.final_answer(q3))

selection = review.last_map()
viz.show_parcel_map(selection, context=data.parcels_df(),
                    title="Riverside Heights — sales by assessment ratio")

I've plotted the 36 sales in Riverside Heights on the map, colored by assessment ratio. The map shows a significant problem: 30 of the parcels are assessed below the IAAO standard range (shown in the cooler colors), meaning they're undervalued, while only 3 are above the range. This suggests systematic underassessment in Riverside Heights that should be investigated further.


That map is **hand-drawn SVG**, and it makes no network request of any kind.

That is not a stylistic preference. A `folium` map pulls in ten external
files — and `leaflet.js` itself is one of them, so on dead conference wifi you
do not get a map missing its streets, you get no map. It would also stop this
notebook's saved output from drawing on GitHub.

There is a second reason, and it is the better one. Every parcel here is
invented. Drawing invented parcels over real streets quietly claims they are
real properties at real addresses, and this repository promises they are not.

⚠️ **One thing worth knowing if you build your own app.** This drew here with
no trouble because Jupyter renders HTML as given. Streamlit does not —
`st.html()` strips every SVG element on the way through, silently, leaving the
card and deleting the picture. The app uses an iframe instead. It cost an
afternoon; it is written up in `CLAUDE.md`.

---

## 🎬 That's Section 9

**What just happened**

1. A ratio study is arithmetic — three tests, checkable by hand. *(Acts 1–2)*
2. Hill Valley passes all three. **Three of its six neighborhoods do not, and
   no two fail the same test.** *(Act 2)*
3. A model can write its own SQL against the roll, read-only. *(Act 3)*
4. **It cannot be trusted with the statistics** — asked to do them in SQL it
   ranked the county's worst neighborhood as its best. So the statistics live
   in a tool. *(Act 4)*
5. Tools that take a query compose, so the assistant answers questions nobody
   wrote a report for. *(Acts 5–6)*

> ### The agent decides what to ask. The tool decides what is correct.

**And the twin notebook agrees with this one.**
The 4B model running on a laptop gets these same numbers, because `ratio_study` computed them either way — not the model.
Sections 5 and 7 both ended with the two models behaving differently. Here
they cannot, and that is the strongest argument in this repository for putting
your domain knowledge in tools: **it is what lets a small model be enough.**

Which is exactly where Section 3 goes next. 🚗💨

---

# 🔬 EXPLORE ON YOUR OWN

Everything below here is off the live path. Nothing depends on it.

## 7. 🏠 Point it at your own county

This is the part to take to your IT department. The ratio study does not know
anything about Hill Valley — it takes a table with an `assessed_value` column
and a `sale_price` column and does arithmetic.

If you can get your sales into a DataFrame with those two columns, everything
in Acts 1 and 2 works on your data today.

In [16]:
import pandas as pd

# Twenty sales from a county that is not Hill Valley. Whatever your office
# has -- a CSV export, a database query, a spreadsheet somebody maintains.
your_sales = pd.DataFrame({
    "sale_price": [
        285_000, 310_000, 262_500, 401_000, 355_000,
        298_000, 332_000, 275_000, 368_000, 412_000,
        244_000, 268_000, 225_000, 310_000, 289_000,
        252_000, 276_000, 233_000, 301_000, 258_000],
    "assessed_value": [
        287_800, 285_200, 286_100, 372_900, 351_400,
        315_900, 315_400, 305_200, 357_000, 424_400,
        185_400, 235_800, 177_800, 260_400, 208_100,
        216_700, 223_600, 209_700, 231_800, 214_100],
    "neighborhood": ["Northside"] * 10 + ["Southside"] * 10,
})

print("All twenty together:")
print("   ", ratio.ratio_stats(your_sales)["summary"])
print()
print("Split by neighborhood:")
print(ratio.ratio_by_group(your_sales, "neighborhood")[
    ["neighborhood", "sales_used", "Median ratio", "COD", "PRD",
     "verdict", "failed"]].to_string(index=False))

All twenty together:
    Meets all three IAAO standards on 20 sales: Median ratio 0.91 (PASS); COD 10.4375 (PASS); PRD 0.9926 (PASS).

Split by neighborhood:
neighborhood  sales_used  Median ratio    COD    PRD verdict               failed
   Southside          10        0.8200 5.6006 1.0018    FAIL Level (Median ratio)
   Northside          10        0.9998 5.3990 1.0051    PASS                    -


Twenty invented sales, and the same thing happens in miniature: **together
they meet every IAAO standard. Split in two, Southside is assessed at 0.82 of
market** and has been carrying too little of the levy all year.

That is the entire point of this section, reproduced on data that has nothing
to do with Hill Valley — which is the check worth doing before you trust any
of this on your own roll.

## 8. 💬 Ask it your own question

Change the sentence and run it. The tools take SQL, so anything you can
describe with a `WHERE` clause is answerable — there is no fixed list of
reports.

Things that work: *the five oldest homes in Lyon Estates* · *a ratio study for
condominiums only* · *map the parcels where we came in above the sale price* ·
*which neighborhood has the largest lots*

In [17]:
your_question = "Run a ratio study for homes built after 1980."

mine = agent.ask(assistant, your_question, system=prompts.REVIEW_ASSISTANT)
viz.show_trace(mine["trace"], answer=agent.final_answer(mine))

sales examined,107
sales used,93
sales excluded not arms length,14
sales excluded unusable,0
verdict,FAIL
note,"14 sale(s) excluded as not arm's length, per IAAO - foreclosures and family transfers are not market evidence."
summary,FAILS Level (Median ratio) on 93 sales: Median ratio 0.8829 (FAIL); COD 11.7794 (PASS); PRD 1.011 (PASS).
stratification note,"This selection mixes property classes (C01, R01, R02). IAAO asks for separate studies per class. To limit it to houses, add: WHERE property_class IN ('R01','R02')"


## 9. 📐 PRB — the measure we compute but do not grade

IAAO added the **Price-Related Bias** because PRD has a known weakness: a
couple of extreme sales can swing it. PRB answers the same equity question
more robustly, and it says something you can repeat out loud:

> *"For every doubling of property value, our ratio moves by PRB × 100 percent."*

Hill Valley demonstrates exactly why it exists — by accident.

In [18]:
hd = ratio.ratio_stats(feed[feed.neighborhood == "Hilldale"])

print("Hilldale, the neighborhood with the equity problem:\n")
for test in hd["tests"]:
    if test["metric"] == "PRD":
        print(f"   PRD  {test['value']:>8}   (standard {test['standard']})"
              f"   {test['verdict']}  <- looks mild")
for extra in hd["also_reported"]:
    print(f"   {extra['metric']:<4} {extra['value']:>8}   "
          f"(standard {extra['standard']})   would be {extra['verdict']}")

print()
print("   Hilldale's ratios really do slide from about 1.35 on a cheap home")
print("   to about 0.66 on an expensive one. PRD calls that mild. PRB does not.")

Hilldale, the neighborhood with the equity problem:

   PRD    1.0431   (standard 0.98 to 1.03)   FAIL  <- looks mild
   PRB   -0.4162   (standard -0.05 to 0.05)   would be FAIL

   Hilldale's ratios really do slide from about 1.35 on a cheap home
   to about 0.66 on an expensive one. PRD calls that mild. PRB does not.


So why is PRB not on the scorecard?

**A teaching reason, not a statistical one.** Graded, PRB fails in three of
six neighborhoods — including two that pass PRD — and every neighborhood ends
up failing two tests. The clean story in Act 2, where each troubled
neighborhood fails exactly one test and tells you what to go and fix, turns to
mush.

An office running this for real should absolutely grade it. It is one entry to
move in [`ratio.py`](../src/delorean/ratio.py), and the code says so.

---

### 🕰️ *"Roads? Where we're going, we don't need roads."* 🕰️

### *Where we're going, we don't need spreadsheets either.*

---

**Next:** [`08_almanac_api.ipynb`](08_almanac_api.ipynb) — Section 8, where the
assistant gets your office's policy manual.